# A full training loop

Now we will see how to achieve the same results as we did using the `Trainer` class without actually using it, instead we will implement a training loop from scratch with modern PyTorch best practices.

In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", "mrpc")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Prepare for training

Before actually writing our training loop, we will need to define a few objects.
- The first ones are the dataloaders we will use to iterate over batches.

But before we can define these dataloaders, we need to apply a bit of postprocessing to our `tokenized_datasets`, to take care of some things that the `Trainer` did for us automatically. Specifically, we need to:
- Remove the columns corresponding to values the model does not expect (like the `sentence1` and `sentence2` columns
- Rename the column `label` to `labels` (because the model expects the argument to be named `labels`).
- Set the format of the datasets so they return PyTorch tensors instead of lists.

Our `tokenized_datasets` has one method for each of those steps:

In [2]:
# Remove the columns corresponding to values the model does not expect
tokenized_datasets = tokenized_datasets.remove_columns(['sentence1', 'sentence2', 'idx'])

# Rename the column 'label' to 'labels'
tokenized_datasets = tokenized_datasets.rename_column('label', 'labels')

# Set the format of the datasets so they return PyTorch tensors instead of lists
tokenized_datasets. set_format('torch')

# Updated Columns
tokenized_datasets['train'].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

We can see that the result only has columns that our model will accept:

``` ['labels', 'input_ids', 'token_type_ids', 'attention_mask'] ```

Now that this is done, we can easily define our dataloaders:

In [9]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=8, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=8, collate_fn=data_collator
)

To quickly check there is no mistake in the data processing, we can inspect a batch like this:

In [11]:
for batch in train_dataloader:
    break

{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([8]),
 'input_ids': torch.Size([8, 74]),
 'token_type_ids': torch.Size([8, 74]),
 'attention_mask': torch.Size([8, 74])}

Note that the actual shapes will probably be slightly different for you because we set `shuffle=True` for the training dataloader and we are padding to the maximum length inside the batch.

Now that we're completely finished with data preprocessing, let's turn to the model. We instantiate it exactly as we did in the previous section:

In [12]:
from transformers import AutoModelForSequenceClassification
# num_labels - Number of labels to use in the last layer added to the model, typically for a classification task.
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

model

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

To make sure that everything will go smoothly during training, we pass our batch to this model:

In [13]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(0.6951, grad_fn=<NllLossBackward0>) torch.Size([8, 2])


All Transformers models will return the loss when `labels` are provided, and we also get the logits (two for each input in our batch, so a tensor of size 8 x 2).
- Our model is a sequence classification model with 2 possible labels (binary classification)
- So for each input sequence, the model outputs 2 raw scores (logits) - one for class 0, and for class 1.

We are almost ready to write our training loop! We're just missing two things: an optimizer and a learning rate scheduler.

Becuase we are trying to replicate what the `Trainer` was doing by hand, we will use the same defaults. The optimizer used by the `Trainer` is `AdamW`, which is the same as Adam, but with a twist for weight decay regularization.

In [15]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

optimizer

AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 5e-05
    maximize: False
    weight_decay: 0.01
)

**Modern Optimization Tips**: For even better performance, you can try:
- **AdamW with weight decay:** ``` AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)```
- **8-bit Adam**: Use `bitsandbytes` for memory-efficient optimization
- **Different learning rates:** Lower learning rates (1e-5 to 3e-5) often work better for large models

Finally, the learning rate scheduler used by default is just a linear decay from the maximum value (5e-5) to 0. To properly define it, we need to know the number of training steps we will take, *which is the number of epochs we want to run multiplied by the number of training batches* (which is the length of our training dataloader). 

The `Trainer` uses three epochs by default, so we will follow that:

In [16]:
from transformers import get_scheduler

num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)
print(num_training_steps)

1377


## The training loop

We will want to use the GPU if we have access to one. On a CPU, training might take several hours instead of a couple of minutes).

To do this, we define a `device` we will put our model and our batches on:

In [17]:
import torch

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
device

device(type='cpu')

We are now ready to train! To get some sense of when training will be finished, we add a progress bar over our number of training steps, using the `tqdm` library.

In [19]:
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_training_steps))

model.train()

for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

  0%|          | 0/1377 [00:00<?, ?it/s]

### Modern Training Optimization:
To make your training loops even more efficient, consider:
- **Gradient Clipping** Add ```torch.nn.utils.clip_grad_norm(model.parameters(), max_norm=1.0)``` before ```optimizer.step()```
- **Mixed Precision** Use ```torch.cuda.amp.autocast()``` and ```GradScaler``` for faster training
- **Gradient Accumulation:** Accumulate gradients over multiple batches to simulate larger batch sizes.
- **Checkpointing**: Save model checkpoints periodically to resume training if interrupted.

## The evaluation loop

We will use a metric provided by the Evaluate library. We've already seen the ```metric.compute()``` method, but metrics can actually accumulate batches for us as we go over the prediction loop with the method ```add_batch()```. Once we have accumulated all the batches, we can get the final result with metric.compute().


Here is how to implement all of this in an evaluation loop:

In [20]:
import evaluate

metric = evaluate.load("glue", "mrpc")
model.eval()
for batch in eval_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    metric.add_batch(predictions=predictions, references=batch["labels"])

metric.compute()

{'accuracy': 0.8480392156862745, 'f1': 0.89419795221843}

## Supercharge your training loop with Accelerate

The training loop we defined earlier works fine on a single CPU or GPU. But using the *Accelerate* library, we just a few adjustments we can enable distributed training on multiple GPUs or TPUs.

**Accelerate library** handles the complexity of distributed training, mixed precision, and device placement automatically.


Starting from the creation of the the training and validation dataloaders, here is what our manual training loop looks like:

In [21]:
from accelerate import Accelerator
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification, get_scheduler

accelerator = Accelerator()

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
optimizer = AdamW(model.parameters(), lr=3e-5)

train_dl, eval_dl, model, optimizer = accelerator.prepare(
    train_dataloader, eval_dataloader, model, optimizer
)

num_epochs = 3
num_training_steps = num_epochs * len(train_dl)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dl:
        outputs = model(**batch)
        loss = outputs.loss
        accelerator.backward(loss)

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/1377 [00:00<?, ?it/s]

The first line to add is the import line. The second line instantiates an `Accelerator` object that will look at the enviornment and initialize the proper distributed setup.
    - *Accelerate* handles the device placement for you so you can remove the lines that put the model on the device. (or, if you prefer, change them to use `accelerator.device` instead of device.

The main bulk of the work is done in the line that sends the dataloaders, the model, and the optimizer to `accelerator.prepare()`. This will wrap those objects in the proper container to make sure your distributed training works as intended. The remaining changes to make are removing the line that puts the batch on the `device` (again, if you want to keep this you can just change it to use `accelerator.device`) and replacing `loss.backward()` with `accelerator.backward(loss)`.

In order to benefit from the speed-up offered by Cloud TPUs, HF recommend padding your samples to a fixed length with the `padding='max_length'` and `max_length` arguments of the tokenizer.

Putting the code above in a `train.py` script will make that script runnable on any kind of distributed setup.

To try it out in your distributed setup, run the command:
`accelerate config`

which will prompt you to answer a few questions and dump your answers in a configuration file used by this command: `accelerate launch train.py`

If you want to try this in a Notebook (for instance, to test it with TPUs on Colab), just paste the code in a `training_function()`

In [25]:
from accelerate import Accelerator
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification, get_scheduler

def training_function():
    accelerator = Accelerator()
    
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
    optimizer = AdamW(model.parameters(), lr=3e-5)
    
    train_dl, eval_dl, model, optimizer = accelerator.prepare(
        train_dataloader, eval_dataloader, model, optimizer
    )
    
    num_epochs = 3
    num_training_steps = num_epochs * len(train_dl)
    lr_scheduler = get_scheduler(
        "linear",
        optimizer=optimizer,
        num_warmup_steps=0,
        num_training_steps=num_training_steps,
    )
    
    progress_bar = tqdm(range(num_training_steps))
    
    model.train()
    for epoch in range(num_epochs):
        for batch in train_dl:
            outputs = model(**batch)
            loss = outputs.loss
            accelerator.backward(loss)
    
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            progress_bar.update(1)

and run a last cell with:

In [30]:
from accelerate import notebook_launcher

notebook_launcher(training_function, num_processes=1)

metric = evaluate.load("glue", "mrpc")
model.eval()
for batch in eval_dataloader:
    batch = {k: v.to(accelerator.device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    metric.add_batch(predictions=predictions, references=batch["labels"])

metric.compute()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Launching training on MPS.


  0%|          | 0/1377 [00:00<?, ?it/s]

{'accuracy': 0.8725490196078431, 'f1': 0.9090909090909091}

## Next Steps and Best Practices

Now that you've learned how to implement training from scratch, here are some additional considerations for production use:

**Model Evaluation**: Always evalute your model on multiple metrics, not just accuracy. Use the *Evaluate* library for comprehensive evaluation.

**Hyperparameter Tuning**: Consider using libraries like Optuna or Ray Tune for systematic hyperparameter optimization.

**Model Monitoring:** Track training metrics, learning curves, and validation performance throughout training.

**Model Sharing**: Once trained, share your model on the Hugging Face Hub to make it available to the community.

**Efficiency**: For large models, consider techniques like gradient checkpointing, parameter-efficient fine-tuning(LoRA, AdaLoRA), or quantization methods.

This concludes the deep dive into fine-tuning with custom training loops. The skiils you've learned here will serve you well when you need full control over the training process or want to implement custom training logic that goes beyond what the `Trainer` API offers.

## Key Takeaways:

- Manual training loops give you complete control but require understanding of the proper sequence: forward -> backward -> optimizer step -> scheduler step -> zero gradients
- AdamW with weight decay is the recommended optimizer for transformer models
- Always use `model.eval()` and `torch.no_grad()` during evaluation for correct behavior and efficiency.
- `Accelerate library` makes disributed training accessible with minimal code changes.
- Device management (moving tensors to GPU/CPU) is crucial for PyTorch operations.
- Modern techniques like mixed precision, gradient accumulation, and gradient clipping can significantly improve traning efficiency.